In [ ]:
pip install streamlit google-genai pypdf reportlab pandas requests beautifulsoup4

In [ ]:
%%writefile app.py
import os
import re
import json
import requests
import streamlit as st
from pypdf import PdfReader
from bs4 import BeautifulSoup
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

from google import genai
from google.genai import types

# ==========================================
# 1. PAGE CONFIGURATION & STYLING
# ==========================================
st.set_page_config(
    page_title="EduPolicy AI - Governance Intelligence Dashboard",
    page_icon="📜",
    layout="wide"
)

st.title("📜 EduPolicy AI: Governance & Regulatory Intelligence Platform")
st.caption("Automated Topic Classification, Chronology Mapping, Risk Detection & Stakeholder Impact Assessment powered by Gemini")

# ==========================================
# 2. HELPER FUNCTIONS: INGESTION & PREPROCESSING
# ==========================================
def extract_text_from_pdf(pdf_file) -> str:
    """Extracts and cleans raw text from an uploaded PDF file."""
    reader = PdfReader(pdf_file)
    extracted_text = ""
    for page in reader.pages:
        text = page.extract_text()
        if text:
            extracted_text += text + "\n"
    return clean_text(extracted_text)

def extract_text_from_url(url: str) -> str:
    """Fetches and cleans text content from a web page or online notice."""
    headers = {"User-Agent": "Mozilla/5.0"}
    res = requests.get(url, headers=headers, timeout=15)
    res.raise_for_status()
    soup = BeautifulSoup(res.content, "html.parser")

    for script in soup(["script", "style", "nav", "footer"]):
        script.extract()

    text = soup.get_text(separator="\n")
    return clean_text(text)

def clean_text(text: str) -> str:
    """Preprocesses and normalizes raw administrative document text."""
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# ==========================================
# 3. REPORTLAB PDF EXPORT GENERATOR
# ==========================================
def generate_downloadable_pdf_report(analysis_dict: dict, filename="Policy_Impact_Report.pdf"):
    """Generates a downloadable PDF summary report using ReportLab."""
    pdf_path = os.path.join("/tmp", filename) if os.name != 'nt' else filename
    doc = SimpleDocTemplate(pdf_path, pagesize=letter)
    styles = getSampleStyleSheet()
    story = []

    title_style = ParagraphStyle('TitleStyle', parent=styles['Heading1'], fontSize=18, leading=22, spaceAfter=12)
    heading_style = ParagraphStyle('HeadingStyle', parent=styles['Heading2'], fontSize=14, leading=18, spaceBefore=10, spaceAfter=6)
    body_style = ParagraphStyle('BodyStyle', parent=styles['Normal'], fontSize=10, leading=14, spaceAfter=6)

    story.append(Paragraph("<b>Policy Intelligence Report</b>", title_style))
    story.append(Paragraph(f"<b>Category:</b> {analysis_dict.get('topic_category', 'N/A')}", body_style))
    story.append(Spacer(1, 10))

    story.append(Paragraph("<b>1. Executive Summary</b>", heading_style))
    story.append(Paragraph(analysis_dict.get('easy_summary', 'N/A'), body_style))
    story.append(Spacer(1, 10))

    story.append(Paragraph("<b>2. Stakeholder Impact Report</b>", heading_style))
    story.append(Paragraph(f"<b>Students:</b> {analysis_dict.get('stakeholder_impact', {}).get('students', 'N/A')}", body_style))
    story.append(Paragraph(f"<b>Faculty:</b> {analysis_dict.get('stakeholder_impact', {}).get('faculty', 'N/A')}", body_style))
    story.append(Paragraph(f"<b>Institutions:</b> {analysis_dict.get('stakeholder_impact', {}).get('institutions', 'N/A')}", body_style))
    story.append(Paragraph(f"<b>Administrators:</b> {analysis_dict.get('stakeholder_impact', {}).get('administrators', 'N/A')}", body_style))
    story.append(Spacer(1, 10))

    story.append(Paragraph("<b>3. Timeframe Horizon Assessment</b>", heading_style))
    story.append(Paragraph(f"<b>Short Term (0-1 yr):</b> {analysis_dict.get('timeframe_impact', {}).get('short_term', 'N/A')}", body_style))
    story.append(Paragraph(f"<b>Medium Term (1-5 yrs):</b> {analysis_dict.get('timeframe_impact', {}).get('medium_term', 'N/A')}", body_style))
    story.append(Paragraph(f"<b>Long Term (>5 yrs):</b> {analysis_dict.get('timeframe_impact', {}).get('long_term', 'N/A')}", body_style))
    story.append(Spacer(1, 10))

    doc.build(story)
    return pdf_path

# ==========================================
# 4. GEMINI API MULTI-TASK ANALYSIS ENGINE
# ==========================================
def analyze_policy_with_gemini(document_text: str, api_key: str) -> dict:
    """Uses Google Gemini API to analyze policy text and return structured JSON output."""
    client = genai.Client(api_key=api_key)

    system_instruction = (
        "You are an expert higher education policy analyst specializing in UGC, AICTE, NAAC, NIRF, and Ministry of Education regulations. "
        "Analyze the provided educational regulation text and output structured JSON."
    )

    prompt = f"""
Analyze the following regulation document and extract structured insights strictly in JSON format matching the given structure:

{{
  "topic_category": "One of: Accreditation, Scholarship, Curriculum, Faculty Policy, Examination, Admissions, Institutional Governance, Other",
  "easy_summary": "10 to 20 line easy-to-read summary explaining what the regulation actually means for a normal student, faculty member, or institution.",
  "chronology_history": [
    {{
      "year_or_phase": "Year or phase name",
      "event": "Description of past circular, amendment, committee, or framework"
    }}
  ],
  "stakeholder_impact": {{
    "students": "How students are affected (benefits or constraints)",
    "faculty": "How faculty members are affected (workload, research, API scores)",
    "institutions": "Impact on colleges/universities (compliance, infrastructure, costs)",
    "administrators": "Impact on administrative processes and deadlines",
    "accreditation_compliance_teams": "Impact on NAAC/NIRF metrics or regulatory reporting"
  }},
  "sentiment_and_risk": {{
    "controversial_areas": "Disputed or sensitive clauses",
    "implementation_bottlenecks": "Key readiness challenges faced by institutions"
  }},
  "timeframe_impact": {{
    "short_term": "0 to 1 year immediate compliance requirements",
    "medium_term": "1 to 5 years operational and structural changes",
    "long_term": "Greater than 5 years strategic impact"
  }},
  "positives": ["Point 1", "Point 2"],
  "negatives": ["Point 1", "Point 2"]
}}

Document Content:
{document_text[:12000]}
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            response_mime_type="application/json",
            temperature=0.1
        )
    )

    return json.loads(response.text)

# ==========================================
# 5. SIDEBAR CONFIGURATION
# ==========================================
with st.sidebar:
    st.header("⚙️ Configuration")

    gemini_api_key = st.text_input(
        "Gemini API Key",
        type="password",
        value=os.getenv("GEMINI_API_KEY", "AQ.Ab8RN6KwmWuDLn5QcS8JLyTLrwKXxRY0FJBSk5QWAPGzzDevfA"),

    )

    st.divider()
    st.header("📥 Ingest Policy Document")

    input_method = st.radio("Choose Input Source", ["Upload PDF", "Paste URL", "Direct Text Input"])

    doc_text = ""

    if input_method == "Upload PDF":
        uploaded_file = st.file_uploader("Upload Circular PDF", type=["pdf"])
        if uploaded_file:
            with st.spinner("Extracting text from PDF..."):
                doc_text = extract_text_from_pdf(uploaded_file)
                st.success(f"Extracted {len(doc_text.split())} words.")

    elif input_method == "Paste URL":
        url_input = st.text_input("Official Circular / Notice URL")
        if url_input and st.button("Fetch Web Document"):
            with st.spinner("Fetching document from URL..."):
                try:
                    doc_text = extract_text_from_url(url_input)
                    st.success(f"Fetched {len(doc_text.split())} words from webpage.")
                except Exception as e:
                    st.error(f"Failed to fetch URL: {e}")

    elif input_method == "Direct Text Input":
        doc_text = st.text_area("Paste Circular Text Here", height=200)

    st.divider()
    analyze_btn = st.button("🚀 Analyze Regulation", type="primary", use_container_width=True)

# ==========================================
# 6. DASHBOARD ANALYSIS & DISPLAY
# ==========================================
if analyze_btn:
    if not gemini_api_key:
        st.error("Please enter your Gemini API Key in the sidebar.")
    elif not doc_text:
        st.warning("Please upload a PDF, enter a valid URL, or paste text to analyze.")
    else:
        with st.spinner("Analyzing document structure, classifying topics, and mapping impacts via Gemini..."):
            try:
                analysis = analyze_policy_with_gemini(doc_text, gemini_api_key)
                st.session_state["analysis_result"] = analysis
                st.success("Analysis Complete!")
            except Exception as e:
                st.error(f"Error during Gemini processing: {e}")

# Render results from session state
if "analysis_result" in st.session_state:
    res = st.session_state["analysis_result"]

    col1, col2 = st.columns([1, 2])
    with col1:
        st.metric("Regulation Category", res.get("topic_category", "Unclassified"))
    with col2:
        pdf_path = generate_downloadable_pdf_report(res)
        with open(pdf_path, "rb") as f:
            st.download_button(
                label="📥 Download Full Summary & Impact Report (PDF)",
                data=f,
                file_name="EduPolicy_AI_Summary_Report.pdf",
                mime="application/pdf",
                use_container_width=True
            )

    st.divider()

    tab_summary, tab_timeline, tab_stakeholders, tab_risks = st.tabs([
        "📝 AI Summary & Overview",
        "⏳ Chronology & Policy Timeline",
        "👥 Stakeholder Impact Report",
        "⚠️ Risk, Positives & Negatives"
    ])

    with tab_summary:
        st.subheader("💡 10–20 Line Plain English Summary")
        st.info(res.get("easy_summary", "No summary generated."))

        st.divider()
        st.subheader("📅 Timeframe Impact Horizon")
        col_s, col_m, col_l = st.columns(3)
        with col_s:
            st.markdown("### Short Term (0–1 Year)")
            st.write(res.get("timeframe_impact", {}).get("short_term", "N/A"))
        with col_m:
            st.markdown("### Medium Term (1–5 Years)")
            st.write(res.get("timeframe_impact", {}).get("medium_term", "N/A"))
        with col_l:
            st.markdown("### Long Term (>5 Years)")
            st.write(res.get("timeframe_impact", {}).get("long_term", "N/A"))

    with tab_timeline:
        st.subheader("📜 Historical Context & Predecessor Mapping")
        timeline = res.get("chronology_history", [])
        if timeline:
            for item in timeline:
                with st.container(border=True):
                    st.markdown(f"**{item.get('year_or_phase', 'Past Event')}**")
                    st.write(item.get("event", "N/A"))
        else:
            st.info("No explicit historical predecessor references detected in this document.")

    with tab_stakeholders:
        st.subheader("👥 Stakeholder Matrix Analysis")
        sh = res.get("stakeholder_impact", {})

        col_a, col_b = st.columns(2)
        with col_a:
            st.markdown("#### 🎓 Students")
            st.write(sh.get("students", "N/A"))

            st.markdown("#### 👨‍🏫 Faculty Members")
            st.write(sh.get("faculty", "N/A"))

            st.markdown("#### 📋 Accreditation & Compliance Teams")
            st.write(sh.get("accreditation_compliance_teams", "N/A"))

        with col_b:
            st.markdown("#### 🏛️ Institutions & Colleges")
            st.write(sh.get("institutions", "N/A"))

            st.markdown("#### ⚙️ Administrators")
            st.write(sh.get("administrators", "N/A"))

    with tab_risks:
        st.subheader("⚠️ Sentiment & Risk Detection")
        sr = res.get("sentiment_and_risk", {})
        st.warning(f"**Controversial / Sensitive Areas:**\n{sr.get('controversial_areas', 'None detected')}")
        st.error(f"**Implementation Bottlenecks & Readiness Challenges:**\n{sr.get('implementation_bottlenecks', 'None detected')}")

        st.divider()
        st.subheader("⚖️ Positives & Negatives Analysis")
        col_pos, col_neg = st.columns(2)

        with col_pos:
            st.markdown("### ✅ Positives & Benefits")
            for pos in res.get("positives", []):
                st.markdown(f"* {pos}")

        with col_neg:
            st.markdown("### ❌ Constraints & Challenges")
            for neg in res.get("negatives", []):
                st.markdown(f"* {neg}")
else:
    st.info("👈 Connect your Gemini API Key and upload/paste a document in the sidebar to begin analysis.")

In [ ]:
!pip install python-docx
import io
import docx
import pypdf
import streamlit as st


def extract_text_from_pdf(file_bytes: bytes) -> str:
    """Extract text from PDF pages using pypdf."""
    reader = pypdf.PdfReader(io.BytesIO(file_bytes))
    text_chunks = []
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text_chunks.append(page_text)
    return "\n".join(text_chunks)


def extract_text_from_docx(file_bytes: bytes) -> str:
    """Extract text from Word documents using python-docx."""
    doc = docx.Document(io.BytesIO(file_bytes))
    text_chunks = [
        paragraph.text for paragraph in doc.paragraphs if paragraph.text.strip()
    ]
    return "\n".join(text_chunks)


def extract_text_from_txt(file_bytes: bytes) -> str:
    """Extract text from plain text files."""
    return file_bytes.decode("utf-8", errors="ignore")


def parse_document(uploaded_file) -> str | None:
    """Safely resets file buffer and parses content based on extension."""
    # CRITICAL: Always reset the file pointer to the beginning of the stream
    uploaded_file.seek(0)
    file_bytes = uploaded_file.read()

    file_extension = uploaded_file.name.split(".")[-1].lower()

    try:
        if file_extension == "pdf":
            return extract_text_from_pdf(file_bytes)
        elif file_extension in ["docx", "doc"]:
            return extract_text_from_docx(file_bytes)
        elif file_extension == "txt":
            return extract_text_from_txt(file_bytes)
        else:
            st.error(f"Unsupported file format: .{file_extension}")
            return None
    except Exception as e:
        st.error(f"Error parsing file: {e}")
        return None


# --- Streamlit UI ---
st.title("📥 Ingest Policy Document")

uploaded_file = st.file_uploader(
    "Choose a Policy Document",
    type=["pdf", "docx", "txt"],
    help="Upload a PDF, Word Document, or TXT file to extract text.",
)

if uploaded_file is not None:
    with st.spinner("Extracting document content..."):
        extracted_text = parse_document(uploaded_file)

    if extracted_text is not None:
        # Calculate clean word count
        words = extracted_text.split()
        word_count = len(words)

        if word_count == 0:
            st.error("⚠️ **0 words extracted!**")
            st.warning(
                "This document appears to be a **scanned image PDF** or password-protected document. "
                "Standard PDF text readers cannot parse pixel images without OCR (Optical Character Recognition)."
            )
        else:
            st.success(
                f" Successfully extracted **{word_count:,} words** from `{uploaded_file.name}`"
            )

            # Display a preview of the text
            st.subheader("Document Preview")
            preview_length = 1500
            st.text_area(
                label="Extracted Text",
                value=extracted_text[:preview_length]
                + ("..." if len(extracted_text) > preview_length else ""),
                height=250,
            )


In [ ]:
!pkill -f streamlit
!pkill -f lt

In [ ]:
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
!streamlit run app.py --server.enableCORS false --server.enableXsrfProtection false &> /dev/null &

In [ ]:
!cloudflared tunnel --url http://localhost:8501

In [ ]:
import subprocess
import time

# 1. Ensure Streamlit runs in the background
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])

# 2. Wait for Streamlit to initialize
print("Starting Streamlit server...")
time.sleep(5)

# 3. Launch Cloudflare Tunnel directly
!cloudflared tunnel --url http://localhost:8501

In [ ]:
!./cloudflared tunnel --url http://localhost:8501

In [ ]:
import subprocess
import time
import re

# Start cloudflared in the background on port 8501 (default Streamlit port)
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

# Wait a few seconds for the link to generate
time.sleep(5)

# Read logs to extract the trycloudflare.com URL
for line in iter(tunnel.stderr.readline, ''):
    if "trycloudflare.com" in line:
        url = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if url:
            print(" Click your app link here:", url.group(0))
            break